# SIRCA — Equipe E — Ciclo 2

Este notebook reúne as etapas de tratamento da base, teste do componente e conferência das consultas. Comece pela primeira célula e siga na ordem. Cada bloco explica o que será feito.

A base é enviada durante a execução e deve permanecer no ambiente autorizado pelo professor. O código foi preparado com apoio de ChatGPT/Codex, conforme a declaração no PDF.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile
import zipfile

checkout = Path('/content/RIAN-1PROJETO')

if not checkout.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/gomesrian241-tech/RIAN-1PROJETO.git', str(checkout)],
        check=True,
    )

RAIZ = checkout / 'sirca'
os.chdir(RAIZ)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
    check=True,
)
print('Ambiente preparado. Continue para os testes.')


## 1. Conferir as funções sem carregar a base

Aqui são testadas as buscas, a normalização dos campos, as pilhas, as filas e as regras dos lançamentos. O resultado esperado é `49 passed, 2 deselected`. Os dois testes restantes dependem da carga e serão executados na próxima etapa.


In [ ]:
subprocess.run([sys.executable,'-m','pytest','componente/testes','-q','-k','not test_csv_igual_modelo_etl and not test_carga_balanco_sem_descartes_ocultos'],check=True)


## 2. Carregar os arquivos do caso

Execute a célula abaixo e selecione o arquivo `SIRCA_EQUIPE_E_CICLO_2.zip`. Também é possível selecionar os três arquivos originais juntos: a planilha operacional e os dois CSV.

Serão usados apenas os três arquivos necessários ao tratamento. Eles ficam no ambiente temporário da sessão. Se a sessão for encerrada, será preciso enviá-los novamente.


In [ ]:
from google.colab import files
esperados = {'Instituto_Semear_planilhas_operacionais_2024-2026.xlsx','frequencia_export_sistema_antigo.csv','doacoes_2026.csv'}
brutos = RAIZ / 'dados/brutos'
brutos.mkdir(parents=True,exist_ok=True)
with tempfile.TemporaryDirectory() as temporario:
    import os
    anterior = Path.cwd()
    try:
        os.chdir(temporario)
        enviados = files.upload()
        encontrados = {}
        for nome,conteudo in enviados.items():
            if Path(nome).name in esperados:
                chave = Path(nome).name
                if chave in encontrados: raise ValueError('Arquivo duplicado: '+chave)
                encontrados[chave] = conteudo
            elif nome.lower().endswith('.zip'):
                import io
                with zipfile.ZipFile(io.BytesIO(conteudo)) as pacote:
                    for item in pacote.infolist():
                        chave = Path(item.filename).name
                        if chave in esperados:
                            if chave in encontrados: raise ValueError('Arquivo duplicado: '+chave)
                            encontrados[chave] = pacote.read(item)
        faltantes = esperados - encontrados.keys()
        if faltantes: raise ValueError('Faltam: '+', '.join(sorted(faltantes)))
        for nome in esperados:
            (brutos/nome).write_bytes(encontrados[nome])
    finally:
        os.chdir(anterior)
print('Três arquivos disponíveis para tratamento; conteúdo não exibido.')


## 3. Tratar os dados e conferir a carga

Esta etapa padroniza os campos, registra as conversões de presença e gera o relatório de rejeições. Depois, executa os 51 testes e mostra as quantidades carregadas.

A célula seguinte confere o banco, compara as cinco consultas SQL com os cálculos em Python e calcula o modelo quantitativo. O banco usado nessa conferência é o PostgreSQL embarcado no PGlite. Os tempos humanos e custos do modelo são hipóteses ilustrativas.


In [ ]:
subprocess.run([sys.executable,'scripts/tratar_dados.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','componente/testes','-q'],check=True)
subprocess.run([sys.executable,'componente/src/main.py','--resumo'],check=True)
(RAIZ/'banco/semear_dump.sql').write_text((RAIZ/'banco/01_criar_tabelas.sql').read_text()+'\n'+(RAIZ/'banco/02_carregar_dados.sql').read_text())


In [ ]:
import shutil
if shutil.which('node') is None or shutil.which('npm') is None:
    raise RuntimeError('Node e npm são necessários para conferir PostgreSQL embarcado; use também as instruções locais do README.')
subprocess.run(['npm','install'],check=True)
subprocess.run(['npm','run','test:banco'],check=True)
subprocess.run([sys.executable,'scripts/conferir_resultados.py'],check=True)
subprocess.run([sys.executable,'estudo/calcular.py'],check=True)


## 4. Abrir o menu do componente

O menu permite listar cadastros, buscar por código ou nome completo, consultar históricos e organizar a fila. Também é possível registrar uma presença e usar as opções de desfazer e refazer.

Os novos lançamentos ficam na memória da sessão. Use a opção **11** para exportá-los, se necessário, e a opção **0** para sair.


In [ ]:
%run componente/src/main.py


## 5. Separar os arquivos da entrega

Após a execução, os arquivos ficam nestas pastas:

- `dados/tratados/`: registros tratados, rejeições e conversões.
- `evidencias/`: relatórios de carga, testes do banco e conferência das consultas.
- `banco/resultados/`: resultados das consultas PN-01 a PN-05.
- `banco/02_carregar_dados.sql` e `banco/semear_dump.sql`: carga e restauração do banco.

Guarde esses arquivos no local restrito autorizado pelo professor e informe os caminhos no PDF. A planilha preenchida e o PDF estão no pacote técnico da equipe; este notebook não os recria.

Antes de entregar, confira os links, a identificação dos integrantes, os papéis, as atas e as evidências de participação. Revise também as saídas do notebook antes de compartilhar uma cópia, pois elas podem conter registros da base.
